

# Part 2. Lifetime exposure

Functions from dem4cli for:

- GMT mapping : this is done year-to-year by matching smoothed GMT timeseries of your original simulation with target stylized GMT timeseries
- Land fraction exposed / average exceedances per year: Note - If your data is binary you will get the annual fraction of land exposed, but if it is not binary you will get the area-weighted average number of exceedances
- Lifetime exposure : per region/country, birthyear and GMT stylized pathways

To do / issues:
- add example smoothing da
- add back 'raw' RCP in le and lfe output if you pick to smooth the data? 
- region_names if subnational make them a bit more helpful! latin names not the acronyms

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import pickle as pk
from scipy import interpolate
import regionmask
import glob, os, re, sys
import openpyxl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings
from math import ceil 
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 20)
%matplotlib inline 

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sys.path.append('../..') # location of dem4cli package
#import demographics4climate as dem4cli # to call them as dem4cli.function 
from dem4cli import * # to call fxns directly


In [2]:
flags

{'version': 2,
 'pop_resolution': 0.1,
 'GMT_mapping': 'year_to_year',
 'cohort_sizes_source': 'UNWPP2024',
 'countrymask': 'shapefile'}

In [3]:
bbox_indiaws = [ 2.00, 40.00, 66.00, 100.00 ]

## 1) Shapefile preprocessing

Use shapefile provided by Paresh

In [4]:
# Load new shapefile, explore it

# Shapefile with polygons to be used in this exercise
gdf_worldadmin = gpd.read_file('/data/brussel/vo/000/bvo00012/vsc11359/data_climakid/Shapefiles/WorldAdmin_V3/WordlAdminV3.shp')

print(gdf_worldadmin.columns)
print(gdf_worldadmin[["GMI_CNTRY"]].head())


DataSourceError: /data/brussel/vo/000/bvo00012/vsc11359/data_climakid/Shapefiles/WorldAdmin_V3/WordlAdminV3.shp: No such file or directory

In [ ]:
# Remove second entry for countries with double entries

# Shapefile with polygons to be used in this exercise
gdf_worldadmin = gpd.read_file('/data/brussel/vo/000/bvo00012/vsc11359/data_climakid/Shapefiles/WorldAdmin_V3/WordlAdminV3.shp')

# Drop rows where GMI_CNTRY is missing or empty 
gdf_worldadmin = gdf_worldadmin[gdf_worldadmin["GMI_CNTRY"].notna() & (gdf_worldadmin["GMI_CNTRY"] != "")].copy()

# Rename to match the column name in the original shapefile, required to be consistent with pipeline
gdf_worldadmin = gdf_worldadmin.rename(columns={"GMI_CNTRY": "ADM0_A3"})

# Remove duplicates, keep only first occurrence of each country ---
gdf_worldadmin = gdf_worldadmin.drop_duplicates(subset="ADM0_A3", keep="first").reset_index(drop=True)

modified_shapefile = "/data/brussel/vo/000/bvo00012/vsc11359/data_climakid/Shapefiles/WorldAdmin_V3/WorldAdmin_V3_woduplicates.shp"
gdf_worldadmin.to_file(modified_shapefile)
print(f"Updated shapefile written to: {modified_shapefile}")

## 2) Population preprocessing

See part 1 for more details: load and preprocess all demographic data (population, cohort sizes, life expectancy)

In [ ]:
d_countries = preprocess_all_country_data(

    filepath_lifeexpectancy = filepath_lifeexpectancy, # life expectancy data
    start_birthyear=1950,
    end_birthyear=2025,                 # endyear is taken from end_birthyear + max life expectancy

    dir_cohortsizes = dir_cohortsizes,  # cohort size data
    data_source_cohorts='UNWPP2024',
    extend_method='linear',             # note, 'slinear' not implemented for UNWPP2024
    by_sex=False,                       # NOTE by_sex not implemented
                                            
    dir_population= dir_population,     # gridded pop data 
    ssp=2,
    urbanrural=False,                   # NOTE urbanrural not implemented for v2
    bbox = bbox_indiaws,

    filepath_countrymask = modified_shapefile,
    data_source_countrymask = 'shapefile',
    fillcoast=False, 
    fix_smallislands=False,
    
    filepath_world_bank = filepath_world_bank_meta, # metadata 
    filepath_lookuptable = filepath_lookuptable,    # country filtering
    filter_countries=True,
    worldbank_filter=True, 
    )

df_countries = d_countries['info_pop']
gdf_country_borders = d_countries['borders'] 
da_regions = df_countries['region'].unique()
da_population = d_countries['population_map']
df_birthyears = d_countries['birth_years'] # NA
df_life_expectancy_5 = d_countries['life_expectancy_5']
da_cohort_size = d_countries['cohort_size']
countries_regions, countries_mask = d_countries['mask']

In [ ]:
da_life_expectancy = (
    df_life_expectancy_5
    .to_xarray()
    .to_array('country')
    .rename({'Year': 'birth_year'})
    .astype(float)
)

In [ ]:
df_life_expectancy_5

## 3) Load GMT stylized trajs

Load stylized GMT trajectories to emulate with the GMT-remapping. 

Using the options : 
- extension past 2100  with 10yrtrend (also available: 10yrmean, lastyear)
- Smoothing of first decades (OS and noOS currently not smoothed). Also available: no smoothing



In [ ]:
df_GMT_15, df_GMT_20, df_GMT_NDC, df_GMT_OS, df_GMT_noOS, ds_GMT_STS, df_GMT_strj = load_GMT(
    year_start=1950,
    year_end=2119,
    gmt_extend_method='10yrtrend',
    smooth_first_decades=True
)

In [ ]:
fig, ax = plt.subplots()

# Plot the background lines for df_GMT_strj without adding legend entries
for col in df_GMT_strj.columns:
    ax.plot(df_GMT_strj.index, df_GMT_strj[col], alpha=0.3, color='gray')  # no label

labels = ['1.5', '2', 'OS', 'noOS', 'NDC']

ax.set_prop_cycle(None)

for i, df in enumerate([df_GMT_15, df_GMT_20, df_GMT_OS, df_GMT_noOS, df_GMT_NDC]):
    labels[i] = labels[i] + ' (' + df.columns.values[0] + ')'
    df.squeeze().plot(ax=ax, label=labels[i])

plt.legend(loc='upper left')

In [ ]:
warminglevels = [1.5, 2, 2.5, 3, 3.5]
offsets = [-.01, 0, +.05, +.12, +.18]

fig, ax = plt.subplots(figsize=(5,4))

df_GMT_strj.loc[1960:2110].plot(ax=ax, legend=False)

ax.axvline(2100, ls='--', c='gray', lw=.8)
for lev, off in zip(warminglevels, offsets):
    ax.text(2111, lev+off, str(lev)+'$\degree$', c='gray', fontsize='8', va='center')

plt.ylabel('GMST ($\degree$C)')

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

## 4) Load climate data and GMT-mapping recipe

Equivalent of old load_isimip() function. Loads in metadata with remapping recipe based on each target trajectory, linked to each original simulation

You can account for re-baselining GMTs to a different baseline (e.g. a bias-adjustment period)



In [ ]:
# We will run the load_climate_data function with these arguments:

data_source = 'ISIMIP'
project_phase = 'ISIMIP2B'
GMT_extra_trajectories = [df_GMT_15, df_GMT_20, df_GMT_NDC, df_GMT_OS, df_GMT_noOS]
GMT_extra_trajectories_names = ['15', '20', 'NDC', 'OS', 'noOS']
year_start=1950
year_end = 2119
gmt_anomaly_baseline_period = (1985,2014)
rolling_window=21
min_periods=11
max_diff_valid=.2


# One may choose among those GCMs depending on the data source:
GCMs = ['GFDL-ESM2M', 'HadGEM2-ES', 'IPSL-CM5A-LR', 'MIROC5']  # for ISIMIP 2b indicators
#GCMs = ['MPI-ESM1-2-HR'] # for ISIMIP3b indicators
#GCMs = ['CanESM5'].   # for ClimaKid indicators

# One may choose among those scenarios depending on the data source:
scenarios = ['rcp26', 'rcp60', 'rcp85'] # for ISIMIP 2b indicators
#scenarios = ['ssp370'] # for ISIMIP3b indicators

# One may choose among those indicators depending on the data source:
extremes = ["heatwavedarea"]  # for ISIMIP2b, choose among "heatwavedarea", "tropicalcyclonedarea", "floodedarea", "driedarea", "cropfailedarea", "burntarea"
#extremes = # for ISIMIP3b, choose among "heatwavedarea", "tropicalcyclonedarea", "floodedarea", "driedarea", "cropfailedarea", "burntarea"

# One may choose among those impact model names epending on the data source:
IMs = {f"{extremes[0]}": ['hwmid99']} # e.g. for ISIMIP2B indicators
# IMs = {f"{extremes[0]}": ['hwmid-humidex']}  # e.g. for ISIMIP3B

In [ ]:
if project_phase.lower() == 'isimip2b':

    # This piece of code reads GMST values from GCMs used in ISIMIP2b and stores them in a dataframe saved as csv, in the format required for the rest of the code to work.

    dir_gmst_gcms = '/data/brussel/vo/000/bvo00012/vsc11137/source2suffering/data/isimip/isimip2b/DerivedInputData/globalmeans/tas'
    simulations = ['historical', 'rcp26', 'rcp45', 'rcp60', 'rcp85']
    file_gmst_gcms = '/data/brussel/vo/000/bvo00012/vsc11359/dem4cli/data/gmst_gcms_ISIMIP2b.csv'

    # Initialize an empty DataFrame
    df_long = pd.DataFrame(columns=['year', 'experiment_id', 'source_id', 'tas'])

    for gcm in GCMs:
        for simul in simulations:
            # Get the file path
            file_path = glob.glob(os.path.join(dir_gmst_gcms, gcm, f'tas_*{gcm}_{simul}*fldmean.yearmean.txt'))[0]

            # Read the data
            df = pd.read_csv(file_path, delim_whitespace=True, comment='#', header=None, names=['year', 'tas'])

        # Filter for years up to 2100
        #df = df[df['year'] <= 2100]

            # Add columns for experiment_id and source_id
            df['experiment_id'] = simul
            df['source_id'] = gcm

            # Append to the long DataFrame
            df_long = pd.concat([df_long, df[['year', 'experiment_id', 'source_id', 'tas']]], ignore_index=True)

    # Sort the DataFrame
    df_long = df_long.sort_values(by=['year', 'experiment_id', 'source_id']).reset_index(drop=True)

    # Export the DataFrame to a CSV file
    df_long.to_csv(file_gmst_gcms, index=True)

    print(f"Data exported to {file_gmst_gcms}")

elif project_phase.lower() == 'isimip3b':
    # for Lange et al indicators from ISIMIP3b
    file_gmst_gcms = os.path.join(data_dir, 'gmst-models/gmst_models_1850_2100_fwi.csv')

    # for ClimaKid indicators
    #file_gmst_gcms = os.path.join(data_dir, 'gmst-models/gmst_models_1850_2100_allmodels.csv')

In [ ]:
df_gmst = pd.read_csv(file_gmst_gcms)
print(df_gmst['experiment_id'].unique())
print(df_gmst['source_id'].unique())
#print(df_gmst['year'].unique())
df_gmst

In [ ]:
gmt_anomaly_correction = calc_gmt_anomaly_correction(
    filepath_gmst_obs=os.path.join(data_dir, 'gmst-obs/GCH_time_series_of_annual_global_temperatures_1850-2024_wrt_1991-2020.csv'),
    col='ERA5',
    gmt_anomaly_baseline_period=(1985,2014),)

gmt_anomaly_correction

# since the climate data is bias-adjusted to 1985-2014 rebaseline everything to this 

In [ ]:
d_climate_data_meta = load_climate_data(
    data_source = data_source,
    project_phase = project_phase,
    extremes = extremes,                   # e.g. "FWI95d"
    gcm_names = GCMs,
    impact_model_names = IMs,             
    df_GMT_strj = df_GMT_strj ,                # stylized trajectories
    scenarios = scenarios,
    GMT_extra_trajectories = GMT_extra_trajectories,
    GMT_extra_trajectories_names = GMT_extra_trajectories_names,
    filepath_model_gmst = file_gmst_gcms,
    rolling_window=21,
    min_periods=11,
    gmt_anomaly_baseline_period = (1985,2014),
    gmt_anomaly_correction = gmt_anomaly_correction,
    year_start=1950,
    year_end = 2119,            # 2025 + max life expectancy
    gmt_mapping_method = 'year-to-year', 
    max_diff_valid = .2, 
    )

In [ ]:
print(d_climate_data_meta[1])

In [ ]:
print(d_climate_data_meta.keys())
for key, item in d_climate_data_meta.items():
    print(f"Entry {key} has keys: {list(item.keys())}")
    print(f"  GCM for {key}: {item['gcm']}")
d_climate_data_meta

In [ ]:
df_mapping = pd.DataFrame(data=d_climate_data_meta[1]['ind_RCP2GMT_strj'], columns=df_GMT_strj.columns, index=np.arange(1950,2120) )
df_mapping

# what index of the simulation is mapped, for all GMT levels, to each year in the scenario-GCM combination corresponding to the first dictionary entry

In [ ]:
# PLOTTING
# Visualise the GMT trajectories that can be reconstructed using the remapping procedure

# Exercise: 
# what is being shown on the x-axis, y-axis? Give proper titles for those
# What does each subpanel correspond to?
# What does each line correspond to? 
# Why are some grey and some coloured? Why do those panels to the right have more coloured lines and those to the left less? 
# Where do the differences between the rows come from?
# Save this plot.
# How would these results differ if we look at another extreme indicator?

fig, axes = plt.subplots(4,3,figsize=(16,16),sharey=True)
axes=axes.flatten()

# Adjust the space between rows
plt.subplots_adjust(hspace=0.4)#, wspace=0.3)

# Collect all unique GMT levels (column names) for the legend
all_gmt_levels = df_GMT_strj.columns

# Create a list of proxy artists for the legend
legend_elements = [
    Line2D([0], [0], color='C{}'.format(i % 10), lw=1.5, alpha=0.8, label=level)
    for i, level in enumerate(all_gmt_levels)
]

for i, ax in enumerate(axes):

    i+=1

    df_mapping = pd.DataFrame(data =d_climate_data_meta[i]['ind_RCP2GMT_strj'], columns=df_GMT_strj.columns, index=np.arange(1950,2120) )
    valid = np.array(d_climate_data_meta[i]['GMT_strj_valid'])  # your 0/1 array

    for col, is_valid in zip(df_mapping.columns, valid):
        if is_valid:
            ax.plot(df_mapping.index, df_mapping[col], lw=1.5, alpha=0.8,label=col)
        else:
            ax.plot(df_mapping.index, df_mapping[col], color='lightgray', lw=1.0, alpha=0.6)


    ax.set_title(d_climate_data_meta[i]['gcm']+' '+d_climate_data_meta[i]['scenario'])

# Add a common y-axis label for the entire figure
fig.text(0.06, 0.5, 'Index of simulation used in year', ha='center', va='center', rotation='vertical', fontsize=14)

# Add the legend to the entire figure (outside all subplots)
fig.legend(handles=legend_elements, loc='lower center', ncol=len(all_gmt_levels)//2, bbox_to_anchor=(0.5, 0.02))

## 4) Land fraction exposed

Note! If your data is BINARY this is land fraction exposed. 

If your data is not binary but is N OF EXCEEDANCES PER YEAR, then you will get the area-weighted average n of exceedances per spatial unit

In [ ]:
for i in list(d_climate_data_meta.keys()):
    scenario = d_climate_data_meta[i]['scenario']
    gcm = d_climate_data_meta[i]['gcm']
    print(i, gcm, scenario)

# get the names / experiments of each element of the dict

# for each dict entry you have remapping recipe for each valid target trajectory 

In [ ]:
# EXERCISE: EXPLORE NEW DICT

#d_meta_subset = dict(list(d_climate_data_meta.items())[0:13])

# Create a new dictionary excluding entries where 'scenario' is 'rcp45'
d_meta_subset = {
    outer_key: inner_dict
    for outer_key, inner_dict in d_climate_data_meta.items()
    if inner_dict.get('scenario') != 'rcp45'
    if inner_dict.get('impact_model') != 'watergap2-2e'
    if inner_dict.get('gcm') != 'IPSL-CM5A-LR'
    if inner_dict.get('gcm') != 'MIROC5'
}

# Reindex the dictionary with sequential keys
d_meta_subset = {i + 1: inner_dict for i, inner_dict in enumerate(d_meta_subset.values())}

d_meta_subset

# test just for a subset of the simulations (faster to run)

In [ ]:
# EXERCISE: EXPLORE NEW DICT

for i in list(d_meta_subset.keys()):
    scenario = d_meta_subset[i]['scenario']
    gcm = d_meta_subset[i]['gcm']
    im = d_meta_subset[i]['impact_model']
    extr = d_meta_subset[i]['extreme']
    print(i, gcm, scenario, im, extr)

In [ ]:
ds_lfe_perregion_perrun, ds_lfe_percountry_perrun, region_names = calc_landfraction_exposed(
    d_climate_data_meta, # replace with d_climate_data_meta to run all data
    df_countries, 
    countries_regions, 
    countries_mask, 
    climatedata_dir,
    GMT_labels = df_GMT_strj.columns,
    GMT_extra_trajectories_names = GMT_extra_trajectories_names,
    year_start=1950,
    year_end=2119,
    bbox=bbox_indiaws,
    weights=None,
    areaweighted=True,
    convert_to_binary=False,            # if your data array is number of exceedances per year (not binary) you will get area-weighted average number of exceedances by default
                                        # if you set this to True your data is turned binary and you can get fraction of land area exposed to at least 1 day
    convert_to_binary_threshold=0,      # here you can change the threshold to e.g. get the fraction of land area exposed to at least X days 
    smoothing_window=21                 # window to smooth also the impact data, get only forced signal not internal variability
                                        # turn this to None for default configuration
)


# note: calling this LFE can be misleading if your data is not binary!

In [ ]:
### EXERCISE: What does the previous function generate?

print(ds_lfe_perregion_perrun.GMT)
print(ds_lfe_perregion_perrun.region)
print(region_names)
ds_lfe_perregion_perrun

In [ ]:
# Exercise: create this dictionary of labels

rename_dict = {
    #i: " ".join(["hadgem2-es (h-c)", "rcp60 (hc)"])
    i: f"{d_climate_data_meta[i]['gcm']} {d_climate_data_meta[i]['scenario']}"
    for i in d_climate_data_meta.keys()
}

rename_dict

In [ ]:
# PLOTTING: projections of land fraction exposed to extreme events

# EXERCISE:
# what is being shown on the x-axis, y-axis? 
#   --> Give proper titles at least for the y-axis
# What does each subpanel correspond to?
#   --> add title for each subpanel to reflect that
# These plots are showing results for which countries or regions? 
#   --> Include this information in the y-axis title
# What does each line correspond to? 
#   --> Add a legend with labels for each line
# What do the differences between the lines tell us?
# Save this plot.
# Later in the exercise, show and save results for at least another extreme indicator.


fig, axes = plt.subplots(1,4,sharey=True,figsize=(10,4))

(ds_lfe_perregion_perrun*100).sel(region=6)['landfrac_peryear_perregion_RCP'].to_pandas().T.rename(columns=rename_dict).plot(ax=axes[0], legend=False)
(ds_lfe_perregion_perrun*100).sel(region=6)['landfrac_peryear_perregion_15'].to_pandas().T.rename(columns=rename_dict).plot(ax=axes[1], legend=False)
(ds_lfe_perregion_perrun*100).sel(region=6)['landfrac_peryear_perregion_20'].to_pandas().T.rename(columns=rename_dict).plot(ax=axes[2], legend=False)
(ds_lfe_perregion_perrun*100).sel(region=6)['landfrac_peryear_perregion_NDC'].to_pandas().T.rename(columns=rename_dict).plot(ax=axes[3], legend=False)

labels = ['RCPs', '1.5°C', '2°C',  'NDC']

for i,ax in enumerate(axes):
    ax.set_title(labels[i])
    ax.set_xticklabels([int(t + year_start) for t in ax.get_xticks()])
    ax.set_xlabel('')  # Remove the x-axis title/label for each subplot

axes[0].set_ylabel(f'area-weighted mean \n{extremes[0]} (%) –– all countries');

# Extract the lines and labels from the first subplot 
lines, line_labels = axes[0].get_legend_handles_labels()

# Add a single legend below the subplots, split into columns
fig.legend(
    lines, line_labels,
    loc='lower center',
    ncol=len(line_labels) // 3,  # Split into columns 
    bbox_to_anchor=(0.5, -0.04),  # Adjust the vertical position as needed
    fancybox=True
)

# Adjust the figure to make room for the legend
plt.subplots_adjust(bottom=0.3)

plt.show()

## 5) Lifetime exposure 

Lifetime exposure per run, and aggregated across runs ("multi-model") for original RCP and for target trajectories after remapping. 

also outputs region names and population-weighted average life-expectancy per region (should this be weighted by cohort aged 0 or total population?? - Ask Emma)



In [ ]:
ds_le_percountry_perrun, ds_le_perregion_perrun, region_names, da_life_expectancy_perregion = calc_lifetime_exposure(
   d_climate_data_meta,
   df_countries, 
   countries_regions, 
   countries_mask, 
   climatedata_dir,
   da_population, 
   df_life_expectancy_5,
   da_cohort_size,
   GMT_labels=df_GMT_strj.columns , 
   GMT_extra_trajectories_names=GMT_extra_trajectories_names,
   start_birthyear=1950,
   end_birthyear=2025,
   year_start=1950,
   year_end=2119,
   bbox=bbox_indiaws,
   smoothing_window=21   
    )



In [ ]:
# EXERCISE: what does the previous function generate?

# For each country and model run, this dataset gives the cumulated number of years a typical person from each birth cohort is or will be affected by a heatwave.

ds_le_percountry_perrun

In [ ]:
# EXERCISE: what does the previous function generate?

ds_le_perregion_perrun

In [ ]:
# EXERCISE: what does the previous function generate?

region_names

In [ ]:
# PLOTTING: projections of land fraction exposed to extreme events

# EXERCISE:
# what is being shown on the y-axis this time?
# Is it the same x-axis as before? 
#   --> Give proper axis titles 

# What does each subpanel correspond to?
#   --> add title for each subpanel to reflect that
# These plots are showing results for which countries or regions? 
#   --> Include this information in the y-axis title

# What does each line correspond to? 
#   --> Add a legend with labels for each line
# What do the differences between the lines tell us?

# Save this plot.

# Can you reproduce these results for another region (among the regions defined in this context)?
# And for a specific country?

# Later in the exercise, show and save results for at least another extreme indicator.


fig, axes = plt.subplots(1,4,sharey=True,figsize=(10,4))

(ds_le_perregion_perrun.sel(region=6)['le_perregion_perrun_RCP'].to_pandas().T).rename(columns=rename_dict).plot(ax=axes[0], legend=False)
(ds_le_perregion_perrun.sel(region=6)['le_perregion_perrun_15'].to_pandas().T).rename(columns=rename_dict).plot(ax=axes[1], legend=False)
(ds_le_perregion_perrun.sel(region=6)['le_perregion_perrun_20'].to_pandas().T).rename(columns=rename_dict).plot(ax=axes[2], legend=False)
(ds_le_perregion_perrun.sel(region=6)['le_perregion_perrun_NDC'].to_pandas().T).rename(columns=rename_dict).plot(ax=axes[3], legend=False)

labels = ['RCPs', '1.5°C', '2°C',  'NDC']

for i,ax in enumerate(axes):
    ax.set_title(labels[i])
    ax.set_xlabel('birth year')

axes[0].set_ylabel(f'lifetime exposure (years) \nto {extremes[0]} –– all countries');

# Extract the lines and labels from the first subplot
lines, line_labels = axes[0].get_legend_handles_labels()

# Add a single legend below the subplots, split into columns
fig.legend(
    lines, line_labels,
    loc='lower center',
    ncol=len(line_labels) // 3,  # Split into 2 columns
    bbox_to_anchor=(0.5, -0.15),  # Adjust the vertical position as needed
    fancybox=True
)

# Adjust the figure to make room for the legend
plt.subplots_adjust(bottom=0.2)

plt.show()

## 6) Compute multi-model mean and model spread

Median, mean and spread across remapped / synthetic trajectories obtained from individual runs

In [ ]:
#  area-weighted average (i.e. land fraction exposed if binary)

ds_lfe_perregion_mmm = calc_landfraction_exposed_mmm(
    ds_lfe_perregion_perrun,
    GMT_extra_trajectories_names,
    )


In [ ]:
# lifetime exposure 

ds_le_perregion_mmm = calc_lifetime_exposure_mmm(
    ds_le_perregion_perrun,
    GMT_extra_trajectories_names,
    year_ref=1950
    )

In [ ]:

ds_lfe_percountry_mmm = calc_landfraction_exposed_mmm(
    ds_lfe_percountry_perrun,
    GMT_extra_trajectories_names,
    )

In [ ]:
ds_le_percountry_mmm = calc_lifetime_exposure_mmm(
    ds_le_percountry_perrun,
    GMT_extra_trajectories_names,
    year_ref=1950
    )

## 7) Plot results for multi-model mean and model spread

In [ ]:
region_names

# Note that World = all countries/area included in your bounding box 

In [ ]:
# PLOTTING: lifetime exposure as a heatmap, function of birth year and GMT
# This plot displays results in a similar way as Fig. 2 in Thiery et al. (2021) or Fig.3 of Grant et al. (2025).

# EXERCISE:
# How is the heatmap constructed?
# On which model is it based?
# What are the units?
# And the axes?


plot = (ds_le_perregion_mmm['median_strj']).sel(region=5).plot(
    cbar_kwargs={'label': f"lifetime exposure to {extremes[0]} (years)"},
    cmap='plasma'
    )

# Get the current axes object
ax = plt.gca()

plt.title('all countries')

# Add a proper x-axis title
ax.set_xlabel('birth year') 

# Show the plot
plt.show()

In [ ]:
# PLOTTING: lifetime exposure as a heatmap, function of birth year and GMT

# EXERCISE:
# Plot the same results, but this time express the lifetime exposure as a percentage of the lifetime.
# Hint: we're talking here about expected lifetime.

plot = (((ds_le_perregion_mmm['median_strj']) / da_life_expectancy_perregion)*100).sel(region=5).plot(
    cbar_kwargs={'label': f"lifetime exposure to {extremes} (% lifetime)"},
    cmap='plasma'
    )

# Get the current axes object
ax = plt.gca()

plt.title('all countries')

# Add a proper x-axis title
ax.set_xlabel('birth year') 

plt.show()


In [ ]:
# PLOTTING: multi-model mean projections of land fraction exposed for various scenarios

# EXERCISE: 
# What variable is being shown here?
#   --> Add y-axis title
# What is the x-axis?
#   --> Add x-axis title
# For which region or country?
# What are the 3 scenarios we're looking at?

# Reproduce and save results for other regions, countries, scenarios of interest.
# Reproduce and save results for other indicators of interest.

# === Parameters ===
country_name = 'India'  # <-- change the country here
year_range = np.arange(1950, 1950+len(ds_lfe_percountry_mmm.time_ind))

# === Define the scenarios to plot ===
GMTs = 1.5, 2, 2.6
colors = {0: 'tab:blue', 1: 'tab:green', 2: 'tab:red'}

# === Create the figure ===
plt.figure(figsize=(6,4))

for i,scen in enumerate(GMTs):
    # Extract multi-model mean and standard deviation for this scenario
    mmm = ds_lfe_percountry_mmm[f'mmm_strj'].sel(country=country_name, GMT=scen).rolling(time_ind=10,min_periods=1).mean() 
    std = ds_lfe_percountry_mmm[f'std_strj'].sel(country=country_name, GMT=scen).rolling(time_ind=10,min_periods=1).mean()
    
    # Plot the mean
    plt.plot(
        year_range, 
        mmm, 
        label=f'Scenario {scen}', 
        color=colors[i],
        lw=2
    )
    
    # Plot the uncertainty range (±2σ)
    plt.fill_between(
        year_range,
        mmm - std,
        mmm + std ,
        alpha=0.3,
        label=rf'±σ',
        color=colors[i],
        lw=1
    )

# === Customize the plot ===
plt.title(f"{extremes[0]}: \nmulti-model mean and uncertainty\nCountry: {country_name}")
plt.xlabel("Time")
plt.ylabel(f"Land fraction exposed to {extremes[0]} (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



# just 2 (cold) remapped simulations, so a little jumpy in hot scenario

In [ ]:
# PLOTTING: maps with results at the national level
# Lifetime exposure as a function of GMT level reached in 2100 and birth year
# 
# What variable should be plotted here? Let's take the example of the multi-model mean to start with.
# Write a line of code that plots results for the GMT level reached in 2100 and birth year of your choice. Each of the subpanels should show results for a different combination of GMT reached in 2100 and birth year.


# -------------------- Parameters -------------------- #
GMTs = [1.5, 1.5, 2.6]
birth_years = [1950,2025,2025]
titles = ['a)', 'b)', 'c)']

use_log_scale = False

vmin=0
vmax=30

cmap = plt.cm.Oranges # or OrRd

# -------------------- Projections ---------- #

central_lon, central_lat = 83, 18.5
extent = [66, 100, 2, 40]

proj = ccrs.Orthographic(central_lon, central_lat)

# -------------------- Plot -------------------- #

fig, axes = plt.subplots(1,3, figsize=(12,8),subplot_kw={'projection': proj},  layout='constrained')


for ax, birth_year, GMT, title in zip(axes.flatten(), birth_years, GMTs, titles):


    # -------------------- Extract values from ds_le_percountry -------------------- #
    # Multi-model mean for the selected scenario
    le_mean = ds_le_percountry_mmm[f'mmm_strj'].sel(birth_year=birth_year, GMT=GMT) 

    # Convert to DataFrame
    df_values = le_mean.to_dataframe(name='mean_val').reset_index()

    # -------------------- Merge with country borders -------------------- #
    gdf_plot = gdf_country_borders.reset_index().merge(
        df_values, left_on='name', right_on='country', how='left'
    )

    gdf_plot.to_crs(proj).plot(
    ax=ax,
    column='mean_val',
    cmap=cmap,
    legend=False,
    legend_kwds={'shrink': 0.5},
    missing_kwds={'color': 'lightgrey'},
    vmin=vmin,vmax=vmax
    )

    ax.set_title(title+f' {birth_year} cohort, {GMT}$\degree$C pathway', loc='left', fontweight='bold')
    ax.set_extent(extent)
    gl = ax.gridlines( draw_labels=True,)
    gl.top_labels = False
    gl.right_labels = False
    ax.coastlines(resolution='50m')

    print(gdf_plot['mean_val'].min(), gdf_plot['mean_val'].max())

# colorbar 
foo = ax.collections[0]
cbar_lab = f'lifetime exposure to {extremes[0]} (years)'
cbar = fig.colorbar(foo, extend='both', ax=axes[0:3], location='bottom',shrink=0.3, fraction=0.08, pad=0.02)
cbar.set_label(label=cbar_lab,size=11)
cbar.outline.set_edgecolor('none')
